# Лабораторная работа 3 по предмету Введение в машинное обучение
Выполнил: студент группы 3311 Шарпинский Денис

Датасет: https://www.kaggle.com/competitions/forest-cover-type-prediction/overview

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sampleSubmission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\n" + "="*50)
print("\nTrain info:")
print(train.info())
print("\n" + "="*50)
print("\nFirst 5 rows:")
print(train.head())
print("\n" + "="*50)
print("\nMissing values:")
print(train.isnull().sum().sum())
print("\n" + "="*50)
print("\nTarget distribution:")
print(train['Cover_Type'].value_counts().sort_index())
print("\n" + "="*50)
print("\nNumerical features statistics:")
print(train.describe())

Train shape: (15120, 56)
Test shape: (565892, 55)


Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15120 entries, 0 to 15119
Data columns (total 56 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   Id                                  15120 non-null  int64
 1   Elevation                           15120 non-null  int64
 2   Aspect                              15120 non-null  int64
 3   Slope                               15120 non-null  int64
 4   Horizontal_Distance_To_Hydrology    15120 non-null  int64
 5   Vertical_Distance_To_Hydrology      15120 non-null  int64
 6   Horizontal_Distance_To_Roadways     15120 non-null  int64
 7   Hillshade_9am                       15120 non-null  int64
 8   Hillshade_Noon                      15120 non-null  int64
 9   Hillshade_3pm                       15120 non-null  int64
 10  Horizontal_Distance_To_Fire_Points  15120 non-null  int64
 11  Wil

### Анализ данных

Видно, что данные не содержат пропусков, классы почти идеально сбалансированы, все признаки числовые (кроме id, его мы уберём в дальнейшем), тип почвы закодирован через one-hot encoding, как и признаки зон дикой природы. Сложная обработка не требуется, можно сразу рабивать данные.

In [5]:
X = train.drop(['Id', 'Cover_Type'], axis=1)
y = train['Cover_Type']
test_ids = test['Id']
X_test = test.drop(['Id'], axis=1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

Train set: (12096, 54)
Validation set: (3024, 54)
Test set: (565892, 54)


### Модель

Изначально был выбран Random Forest, но он не дал необходимой точности ( > 0.75 ), так что был выбран Gradient Boosting - модель делает предсказание, после чего старается учесть ошибку первого предсказания и исправить её.

Ключевые гиперпараметры:
- `n_estimators` - количество деревьев (итераций)
- `learning_rate` - вес каждого нового дерева
- `max_depth` - максимальная глубина каждого дерева
- `subsample` - доля данных для обучения каждого дерева
- `min_samples_split/leaf` - ограничения на размер узлов дерева

In [4]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=7, 
                                min_samples_split=10, min_samples_leaf=4, 
                                subsample=0.8, random_state=42, verbose=1)
gb.fit(X_train, y_train)

y_val_pred_gb = gb.predict(X_val)
val_accuracy_gb = accuracy_score(y_val, y_val_pred_gb)

print(f"GradientBoosting Validation Accuracy: {val_accuracy_gb:.4f}")

y_test_pred_gb = gb.predict(X_test)

submission_gb = pd.DataFrame({
    'Id': test_ids,
    'Cover_Type': y_test_pred_gb
})

submission_gb.to_csv('submission_gb.csv', index=False)
print("\nGradientBoosting submission created!")

      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.5911           0.3274            1.01m
         2           1.3770           0.2277            1.04m
         3           1.2164           0.1674            1.04m
         4           1.0845           0.1167            1.03m
         5           0.9792           0.1094            1.03m
         6           0.8915           0.0879            1.03m
         7           0.8164           0.0727            1.03m
         8           0.7546           0.0702            1.02m
         9           0.7000           0.0542            1.02m
        10           0.6535           0.0485            1.02m
        20           0.3900           0.0008           58.80s
        30           0.2886           0.0111           56.77s
        40           0.2283           0.0138           54.67s
        50           0.1863           0.0102           52.74s
        60           0.1541           0.0097           50.65s
       

### Выводы

Датасет оказался практически полностью готовым к обучению модели, так что лабораторная работа оказалась весьма короткой. Был удалён столбец id и обучен Gradient Bossting CLassifier.